In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

BASE_DIR    = Path().resolve().parent
TREC_DIR    = BASE_DIR / 'models' / 'trec'
FIGURES_DIR = BASE_DIR / 'figures'

ALGO_COLORS = {
    'Naive Bayes':         '#1C2B4A',
    'SVM':                 '#0891B2',
    'Random Forest':       '#2563EB',
    'Regressão Logística': '#64748B',
}

print(f'Projeto em : {BASE_DIR}')

In [ ]:
def load(name):
    with open(TREC_DIR / f'{name}.pkl', 'rb') as f:
        return pickle.load(f)

X_train = load('X_train_tfidf')
X_test  = load('X_test_tfidf')
y_train = load('y_train')
y_test  = load('y_test')

print(f'Treino : {X_train.shape}')
print(f'Teste  : {X_test.shape}')
print(f'\nDistribuição no teste:')
print(pd.Series(y_test).value_counts().rename({1: 'spam', 0: 'ham'}))
print(f'\nProporção spam no teste: {y_test.mean():.2%}')

In [ ]:
models = {
    'Naive Bayes':         MultinomialNB(),
    'SVM':                 LinearSVC(random_state=42, max_iter=2000, dual='auto'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
}

results = []
for name, model in models.items():
    print(f'Treinando: {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'Algoritmo': name,
        'Acurácia':  accuracy_score(y_test, y_pred),
        'Precisão':  precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        'F1-score':  f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    })
    slug = name.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c')
    with open(TREC_DIR / f'model_{slug}.pkl', 'wb') as f:
        pickle.dump(model, f)

df_results = pd.DataFrame(results).set_index('Algoritmo')
df_results = df_results.sort_values('F1-score', ascending=False)

print('\n' + '=' * 60)
print('TABELA COMPARATIVA DE DESEMPENHO — TREC 2007 (150 palavras)')
print('=' * 60)
print(df_results.round(4).to_string())
print('=' * 60)

df_results.to_csv(TREC_DIR / 'results.csv')
print('Salvo: models/trec/results.csv')

In [ ]:
from sklearn.model_selection import StratifiedKFold

print('Validação cruzada 10-fold estratificada (F1-score no treino):\n')

cv_results = {}
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
for name, model in models.items():
    m = model.__class__(**model.get_params())
    scores = cross_val_score(m, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25} média={scores.mean():.4f}  std={scores.std():.4f}')

In [5]:
# Célula 5 — Relatórios de classificação detalhados
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'\n{"=" * 50}')
    print(f'  {name}')
    print(f'{"=" * 50}')
    print(classification_report(
        y_test, y_pred,
        target_names=['Ham', 'Spam'],
        digits=4
    ))


  Naive Bayes
              precision    recall  f1-score   support

         Ham     0.8849    0.9907    0.9348      5044
        Spam     0.9948    0.9331    0.9630      9711

    accuracy                         0.9528     14755
   macro avg     0.9399    0.9619    0.9489     14755
weighted avg     0.9573    0.9528    0.9533     14755


  SVM
              precision    recall  f1-score   support

         Ham     0.9916    0.9869    0.9893      5044
        Spam     0.9932    0.9957    0.9944      9711

    accuracy                         0.9927     14755
   macro avg     0.9924    0.9913    0.9919     14755
weighted avg     0.9927    0.9927    0.9927     14755




  Random Forest


              precision    recall  f1-score   support

         Ham     0.9926    0.9869    0.9898      5044
        Spam     0.9932    0.9962    0.9947      9711

    accuracy                         0.9930     14755
   macro avg     0.9929    0.9916    0.9922     14755
weighted avg     0.9930    0.9930    0.9930     14755


  Regressão Logística
              precision    recall  f1-score   support

         Ham     0.9904    0.9784    0.9843      5044
        Spam     0.9888    0.9951    0.9919      9711

    accuracy                         0.9894     14755
   macro avg     0.9896    0.9867    0.9881     14755
weighted avg     0.9894    0.9894    0.9893     14755



In [ ]:
metrics = ['Acurácia', 'Precisão', 'Recall', 'F1-score']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle(
    'Desempenho Comparativo dos Algoritmos — TREC 2007 (recorte 150 palavras)',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, metric in zip(axes, metrics):
    valores = df_results[metric].sort_values(ascending=True)
    cores   = [ALGO_COLORS[a] for a in valores.index]
    bars = ax.barh(valores.index, valores.values, color=cores, edgecolor='white')
    ax.set_xlim(max(0, valores.min() - 0.05), 1.0)
    ax.set_title(metric, fontweight='bold', fontsize=11)
    ax.set_xlabel('Score')
    ax.tick_params(axis='y', labelsize=9)
    for bar, val in zip(bars, valores.values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparative_metrics_trec.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/comparative_metrics_trec.png')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(
    'Matrizes de Confusão — TREC 2007 (recorte 150 palavras)',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Ham', 'Spam'],
        yticklabels=['Ham', 'Spam'],
        linewidths=0.5, linecolor='white'
    )
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices_trec.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/confusion_matrices_trec.png')

In [ ]:
melhor      = df_results['F1-score'].idxmax()
model_best  = models[melhor]
y_pred_best = model_best.predict(X_test)

fp = ((y_pred_best == 1) & (y_test == 0)).sum()
fn = ((y_pred_best == 0) & (y_test == 1)).sum()

print(f'Melhor algoritmo por F1-score: {melhor}')
print(f'\nMétricas:')
print(df_results.loc[melhor].round(4).to_string())
print(f'\nAnálise de erros ({melhor}):')
print(f'  Falsos positivos (ham → spam): {fp}')
print(f'  Falsos negativos (spam → ham): {fn}')
# Falsos negativos são mais custosos: spam não detectado chega ao usuário
print(f'\n  Falsos negativos são mais custosos em segurança —')
print(f'  representam spam que passou pelo filtro.')

In [ ]:
# Célula 9 — Tabela 1 do TG (matplotlib)
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

col_labels = ['Algoritmo', 'Acurácia', 'Precisão', 'Recall', 'F1-score']
cell_data = [
    ['Random Forest',       '0,9930', '0,9932', '0,9962', '0,9947'],
    ['SVM',                 '0,9927', '0,9932', '0,9957', '0,9944'],
    ['Regressão Logística', '0,9894', '0,9888', '0,9951', '0,9919'],
    ['Naive Bayes',         '0,9528', '0,9948', '0,9331', '0,9630'],
]

tbl = ax.table(
    cellText=cell_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 1.6)

NAVY = '#1C2B4A'
for j in range(len(col_labels)):
    cell = tbl[0, j]
    cell.set_facecolor(NAVY)
    cell.set_text_props(color='white', fontweight='bold')

for i in range(len(cell_data)):
    bg = '#F0F4FF' if i % 2 == 0 else '#FFFFFF'
    if i == len(cell_data) - 1:
        bg = '#FEF2F2'
    for j in range(len(col_labels)):
        cell = tbl[i + 1, j]
        cell.set_facecolor(bg)
        cell.set_edgecolor('#E2E8F0')
        if j == 0:
            cell.set_text_props(ha='left')
        if j == 4:
            cell.set_text_props(fontweight='bold')

fig.suptitle(
    'Tabela 1 — Desempenho comparativo dos algoritmos no conjunto de teste\n'
    'TREC 2007 Public Spam Corpus · Recorte de 150 palavras',
    fontsize=11, fontweight='bold', y=0.95,
)
fig.text(0.5, 0.02, 'Fonte: Elaborado pelo autor.', ha='center', fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tabela_resultados.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/tabela_resultados.png')

In [ ]:
# Célula — Gerar tabela de especificações da IA com matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE_DIR    = Path().resolve().parent
FIGURES_DIR = BASE_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# ── Dados ─────────────────────────────────────────────────────────────────────
NAVY  = '#1C2B4A'
TEAL  = '#0891B2'
WHITE = '#FFFFFF'
LIGHT = '#F0F4FF'
GRAY  = '#64748B'

headers = ['Componente', 'Implementação', 'Parâmetros']
rows = [
    ['Naive Bayes',        'MultinomialNB',            'Padrão (alpha=1,0)'],
    ['SVM',                'LinearSVC',                "dual='auto', max_iter=2.000, random_state=42"],
    ['Random Forest',      'RandomForestClassifier',   'n_estimators=200, random_state=42, n_jobs=-1'],
    ['Regressão Logística','LogisticRegression',       'max_iter=1.000, random_state=42, n_jobs=-1'],
    ['Vetorização',        'TfidfVectorizer',          'max_features=10.000, sublinear_tf=True,\nmin_df=2, ngram_range=(1,2)'],
    ['Split',              'train_test_split',         'test_size=0,2, random_state=42, stratify=y'],
    ['Validação',          'StratifiedKFold',          'n_splits=10, shuffle=True, random_state=42'],
]

# ── Figura ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.2))
fig.patch.set_facecolor(WHITE)
ax.axis('off')

# Larguras relativas das colunas
col_widths = [0.18, 0.22, 0.60]

table = ax.table(
    cellText=rows,
    colLabels=headers,
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1],
    colWidths=col_widths,
)

table.auto_set_font_size(False)
table.set_fontsize(10.5)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#E2E8F0')
    cell.PAD = 0.06

    # Cabeçalho
    if row == 0:
        cell.set_facecolor(NAVY)
        cell.set_text_props(color=WHITE, fontweight='bold', fontfamily='monospace' if col > 0 else 'sans-serif')

    # Linha do TF-IDF (row 5) — tem quebra de linha, precisa de altura maior
    elif row == 5:
        cell.set_facecolor(LIGHT)
        cell.set_text_props(fontfamily='monospace' if col > 0 else 'sans-serif', fontsize=9.5)

    # Linhas alternadas
    elif row % 2 == 0:
        cell.set_facecolor(LIGHT)
        cell.set_text_props(fontfamily='monospace' if col > 0 else 'sans-serif')
    else:
        cell.set_facecolor(WHITE)
        cell.set_text_props(fontfamily='monospace' if col > 0 else 'sans-serif')

    # Coluna de componentes — negrito
    if col == 0 and row > 0:
        cell.set_text_props(fontweight='bold', color=NAVY, fontfamily='sans-serif')

# Ajustar altura da linha do TF-IDF (quebra de linha)
for col in range(3):
    table[5, col].set_height(table[5, col].get_height() * 1.8)

table.auto_set_column_width([0, 1, 2])

# ── Título e rodapé ───────────────────────────────────────────────────────────
fig.suptitle(
    'Tabela X — Configurações dos algoritmos e do pipeline de vetorização',
    fontsize=11, fontweight='bold', color=NAVY, y=1.02
)
fig.text(
    0.5, -0.04,
    'Fonte: Elaborado pelo autor.',
    ha='center', fontsize=9, color=GRAY, style='italic'
)

plt.tight_layout()

# ── Salvar ────────────────────────────────────────────────────────────────────
output_path = FIGURES_DIR / 'tabela_especificacoes_ia.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight',
            facecolor=WHITE, edgecolor='none')
plt.show()
print(f"Tabela salva em: {output_path}")

In [ ]:
# Célula — Gerar fluxograma do pipeline com matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE_DIR    = Path().resolve().parent
FIGURES_DIR = BASE_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# ── Paleta ────────────────────────────────────────────────────────────────────
NAVY   = '#1C2B4A'
TEAL   = '#0891B2'
BLUE   = '#2563EB'
WHITE  = '#FFFFFF'
GRAY   = '#64748B'
GREEN  = '#16A34A'

# ── Figura ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor(WHITE)
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, sublabel=None,
             facecolor=NAVY, textcolor=WHITE, fontsize=10):
    box = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.1',
        facecolor=facecolor, edgecolor='#0E7490',
        linewidth=1.5, zorder=2
    )
    ax.add_patch(box)
    if sublabel:
        ax.text(x + w/2, y + h*0.62, label,
                ha='center', va='center',
                fontsize=fontsize, fontweight='bold',
                color=textcolor, zorder=3)
        ax.text(x + w/2, y + h*0.28, sublabel,
                ha='center', va='center',
                fontsize=fontsize - 1.5,
                color=textcolor, alpha=0.85, zorder=3,
                fontstyle='italic')
    else:
        ax.text(x + w/2, y + h/2, label,
                ha='center', va='center',
                fontsize=fontsize, fontweight='bold',
                color=textcolor, zorder=3)

def arrow_h(ax, x1, x2, y):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
        arrowprops=dict(arrowstyle='->', color=TEAL, lw=2, mutation_scale=16), zorder=3)

def arrow_v(ax, x, y1, y2):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
        arrowprops=dict(arrowstyle='->', color=TEAL, lw=2, mutation_scale=16), zorder=3)

# ── ROW 1 (topo): Dataset → Truncagem → Pré-proc → TF-IDF ───────────────────
BOX_W, BOX_H = 2.4, 1.1
Y1 = 7.5
r1 = [
    (0.3,  'Dataset\nTREC 2007',    '73.932 e-mails',                            NAVY),
    (3.15, 'Truncagem',             '150 primeiras palavras',                     TEAL),
    (6.0,  'Pré-processamento',     'lowercase · regex\ntokenização · stopwords', BLUE),
    (8.85, 'Vetorização\nTF-IDF',   'max_features=10.000\nngram_range=(1,2)',     NAVY),
]
for i, (x, label, sub, color) in enumerate(r1):
    draw_box(ax, x, Y1, BOX_W, BOX_H, label, sub, facecolor=color, fontsize=9.5)
    if i < len(r1) - 1:
        arrow_h(ax, x + BOX_W, r1[i+1][0], Y1 + BOX_H/2)

# ── Seta vertical: TF-IDF → Split ────────────────────────────────────────────
tfidf_cx = 8.85 + BOX_W/2
arrow_v(ax, tfidf_cx, Y1, 6.55)

# ── ROW 2: Split (centralizado sob TF-IDF) ───────────────────────────────────
SPLIT_W, SPLIT_H = 2.6, 1.0
SPLIT_X = tfidf_cx - SPLIT_W/2
SPLIT_Y = 5.5
draw_box(ax, SPLIT_X, SPLIT_Y, SPLIT_W, SPLIT_H,
         'Split Estratificado', '80% treino / 20% teste',
         facecolor=GRAY, fontsize=9.5)
split_cx = SPLIT_X + SPLIT_W/2

# ── ROW 3: Conjuntos de Treino e Teste ───────────────────────────────────────
TREINO_X, TREINO_W = 1.0, 2.6
TREINO_Y, TREINO_H = 3.9, 1.0
treino_cx = TREINO_X + TREINO_W/2

TESTE_X, TESTE_W = 10.8, 2.6
TESTE_Y, TESTE_H = 3.9, 1.0
teste_cx = TESTE_X + TESTE_W/2

draw_box(ax, TREINO_X, TREINO_Y, TREINO_W, TREINO_H,
         'Conjunto de Treino', '59.019 e-mails', facecolor=NAVY, fontsize=9.5)
draw_box(ax, TESTE_X, TESTE_Y, TESTE_W, TESTE_H,
         'Conjunto de Teste', '14.755 e-mails', facecolor=NAVY, fontsize=9.5)

# ── Setas do Split → Treino (esquerda) e Split → Teste (direita) ─────────────
# Split → Treino: desce, vai para a esquerda, desce até o topo do box
mid_y_treino = TREINO_Y + TREINO_H + 0.4
ax.plot([split_cx, split_cx], [SPLIT_Y, mid_y_treino],
        color=TEAL, lw=1.5, zorder=2)
ax.plot([split_cx, treino_cx], [mid_y_treino, mid_y_treino],
        color=TEAL, lw=1.5, zorder=2)
arrow_v(ax, treino_cx, mid_y_treino, TREINO_Y + TREINO_H)

# Split → Teste: desce, vai para a direita, desce até o topo do box
mid_y_teste = TREINO_Y + TREINO_H + 0.4
ax.plot([split_cx, teste_cx], [mid_y_teste, mid_y_teste],
        color=TEAL, lw=1.5, zorder=2)
arrow_v(ax, teste_cx, mid_y_teste, TESTE_Y + TESTE_H)

# ── Seta: Treino → Algoritmos ────────────────────────────────────────────────
arrow_v(ax, treino_cx, TREINO_Y, 3.5)

# ── ROW 4: Algoritmos (empilhados verticalmente) ─────────────────────────────
ALGO_W, ALGO_H = 2.0, 0.7
ALGO_X = treino_cx - ALGO_W/2
algo_labels = ['Naive Bayes', 'SVM', 'Random Forest', 'Reg. Logística']
algo_ys = [2.7, 1.95, 1.2, 0.45]

for y, label in zip(algo_ys, algo_labels):
    draw_box(ax, ALGO_X, y, ALGO_W, ALGO_H, label, facecolor=TEAL, fontsize=9)

# Linha vertical convergente à direita dos algos
conv_x = ALGO_X + ALGO_W + 0.3
top_algo_cy = algo_ys[0] + ALGO_H/2
bot_algo_cy = algo_ys[-1] + ALGO_H/2
mid_algo_cy = (top_algo_cy + bot_algo_cy) / 2

for y in algo_ys:
    arrow_h(ax, ALGO_X + ALGO_W, conv_x, y + ALGO_H/2)

ax.plot([conv_x, conv_x], [bot_algo_cy, top_algo_cy],
        color=TEAL, lw=2, zorder=2)
arrow_h(ax, conv_x, 7.2, mid_algo_cy)

# ── Avaliação ─────────────────────────────────────────────────────────────────
AVAL_W, AVAL_H = 2.6, 1.2
AVAL_X = 7.2
AVAL_Y = mid_algo_cy - AVAL_H/2
draw_box(ax, AVAL_X, AVAL_Y, AVAL_W, AVAL_H,
         'Avaliação', 'Acurácia · Precisão\nRecall · F1-score',
         facecolor=GREEN, fontsize=10)

# ── Seta: Teste → Avaliação (em L) ───────────────────────────────────────────
aval_cx = AVAL_X + AVAL_W/2
aval_top = AVAL_Y + AVAL_H

ax.plot([teste_cx, teste_cx], [TESTE_Y, aval_top + 0.35],
        color=TEAL, lw=1.5, zorder=2)
ax.plot([teste_cx, aval_cx], [aval_top + 0.35, aval_top + 0.35],
        color=TEAL, lw=1.5, zorder=2)
arrow_v(ax, aval_cx, aval_top + 0.35, aval_top)

# ── Título e rodapé ───────────────────────────────────────────────────────────
fig.suptitle(
    'Figura X — Pipeline completo de classificação de spam em e-mails',
    fontsize=12, fontweight='bold', color=NAVY, y=0.98
)
fig.text(0.5, 0.01,
    'Fonte: Elaborado pelo autor.',
    ha='center', fontsize=9, color=GRAY, style='italic')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# ── Salvar ────────────────────────────────────────────────────────────────────
output_path = FIGURES_DIR / 'pipeline_fluxograma.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight',
            facecolor=WHITE, edgecolor='none')
plt.show()
print(f"Fluxograma salvo em: {output_path}")